# Verify the Mophongo Pipeline

This notebook runs the standard realistic two-detector F444W/F770W verification setup.  It uses `MockMosaic` with real STPSF/DrizzlePSF products, a two-detector NIRCam LW layout, two MIRI macro positions with 8 phase dithers each, 1000 mixed point/extended sources, and the package's standard diagnostics.

The notebook does not define custom image diagnostic PNGs. PSF matching diagnostics come from `PSF.optimize_matching_kernel_regularization`; source-stage diagnostics come from `Pipeline.diagnose_sources`; flux-recovery plots come from `mophongo.verification.save_flux_recovery_plot`.

This run also applies a fixed F770W-vs-F444W source-position shift and writes the fitted `Scene.plot` diagnostics for the selected scenes.


In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mophongo_mpl_cache")

import matplotlib.pyplot as plt
from astropy.table import Table

ROOT = Path.cwd()
if not (ROOT / "src" / "mophongo").exists():
    if (ROOT.parent / "src" / "mophongo").exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError(f"Could not locate mophongo repo root from {Path.cwd()}")

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from mophongo.verification import (
    DEFAULT_WIENER_REG_GRID,
    build_realistic_two_detector_mock,
    build_wiener_psf_maps,
    run_pipeline_extension_scenario,
    wht_noise_check_from_fits,
)

OUT = ROOT / "examples" / "verify_pipeline_realistic_out"
MOCK_OUT = OUT / "mock"
PSF_DIR = ROOT / "data" / "PSF"
if not PSF_DIR.exists():
    raise FileNotFoundError(f"Expected real PSFs under {PSF_DIR}")

NSRC = 1000
SIGMA_RANGE = (1.0, 5.0)
POINT_SOURCE_FRACTION = 0.10
TEMPLATE_DILATE_SEGMAP = 4
MOCK_DILATE_SEGMAP = 2
SCENARIOS = ("none", "psf_wings")
# Fixed astrometric offset applied while creating the F770W truth scene,
# in native F770W pixel (dx, dy) order. The verification helper restricts
# each axis to +/-1 native F770W pixel. After pipeline upsampling, this
# corresponds to about twice this shift on the F444W grid.
F770W_FIXED_SHIFT_XY = (-0.80, 0.95)
# Extra Gaussian PSF broadening (FWHM arcsec, per filter). None uses the
# package-wide shared default mock_mosaic.DEFAULT_PSF_GAUSSIAN_FWHM_ARCSEC
# ({'f770w': 0.08}) — the SAME constant and operator (gaussian_blur_psf)
# applied by MockMosaic painting, the Wiener PSF/kernel maps, and the
# real-data driver examples/run_770.py. Pass 0.0 or {} to disable.
PSF_BLUR_FWHM_ARCSEC = None
OUT.mkdir(parents=True, exist_ok=True)
OUT

Matplotlib is building the font cache; this may take a moment.
INFO: NumExpr defaulting to 10 threads.


PosixPath('/Users/ivo/Astro/PROJECTS/MOPHONGO/mophongo/mophongo/examples/verify_pipeline_realistic_out')

## Build the Realistic Mock

The mock uses the same layout as the scratch realistic validation: F444W on NRCA5+NRCB5 with six phase dithers, and F770W with two macro pointings aligned to the LW detectors plus eight MIRI phase dithers at each macro position.  The PSF stamp sums are preserved as finite-support throughput metadata.

In [ ]:
mock, paths, noise_info, dpsfs, truth = build_realistic_two_detector_mock(
    MOCK_OUT,
    psf_dir=PSF_DIR,
    nsrc=NSRC,
    sigma_range=SIGMA_RANGE,
    point_source_fraction=POINT_SOURCE_FRACTION,
    snr_range=(10.0, 5000.0),
    seed=42,
    f770w_position_shift_xy=F770W_FIXED_SHIFT_XY,
    psf_gaussian_fwhm_arcsec=PSF_BLUR_FWHM_ARCSEC,
)
plt.close("all")

checks = []
for filt in ("f444w", "f770w"):
    sci = paths[filt]["fits"]
    checks.append(
        wht_noise_check_from_fits(
            sci,
            sci.with_name(sci.name.replace("_sci", "_truth")),
            sci.with_name(sci.name.replace("_sci", "_wht")),
            filter_name=filt,
        )
    )
Table(rows=[c.__dict__ for c in checks])

## Build the Standard Wiener Kernel Diagnostic

The regularization scan is performed once on representative unit-sum PSF shapes using the package PSF diagnostic.  The scan range is the standard `1e-6 ... 0.1` grid.

In [ ]:
psf_maps = build_wiener_psf_maps(
    mock,
    paths,
    dpsfs,
    OUT,
    psf_dir=PSF_DIR,
    reg_grid=DEFAULT_WIENER_REG_GRID,
    kernel_grid_nside=1,
)
print(f"best Wiener lambda = {psf_maps.wiener_lambda:g}")
print(f"F444W average throughput = {psf_maps.source_throughputs.mean():.4f}")
print(f"F770W average throughput = {psf_maps.target_throughputs.mean():.4f}")

## Run the Pipeline Scenarios

Only the two production-relevant template-extension settings are compared here: no extension and PSF-wing completion.  The scenario runner writes the standard source-stage diagnostics and flux-recovery tables/plots.

In [ ]:
results = {}
for scenario in SCENARIOS:
    result = run_pipeline_extension_scenario(
        scenario,
        out_dir=OUT,
        paths=paths,
        noise_info=noise_info,
        truth=truth,
        psf_maps=psf_maps,
        mock_dilate_segmap=MOCK_DILATE_SEGMAP,
        template_dilate_segmap=TEMPLATE_DILATE_SEGMAP,
        fit_astrometry_niter=8,
        fit_background=False,
        source_diagnostic_count=10,
        full_diagnostic_highres_size=3000,
        scene_diagnostic_count=12,
        f770w_position_shift_xy=F770W_FIXED_SHIFT_XY,
        nsrc=NSRC,
        sigma_range=SIGMA_RANGE,
        point_source_fraction=POINT_SOURCE_FRACTION,
    )
    results[scenario] = result

summary = Table(rows=[result.summary for result in results.values()])
summary.write(OUT / "template_extension_summary.csv", overwrite=True)
assert all(bool(v) for v in summary["f770w_shift_recovered_ok"]), summary
summary

## Diagnostic Outputs

The PNG outputs below are generated by the standard package diagnostics and helpers.  There is intentionally no custom image-strip diagnostic in this notebook.

In [ ]:
for path in sorted(OUT.rglob("*.png")):
    print(path.relative_to(OUT))

## Catalog Flux Convention

`flux_2` in the pipeline catalog is the fitted unit-template model amplitude.  `flux_2_total` is the filter-throughput corrected total-flux estimate, using one average finite-support PSF correction per filter.

In [ ]:
cols = ["id", "flux_true", "throughput_2", "flux_2_model", "flux_2_total", "ratio_2", "err_pred_2_total"]
results["psf_wings"].source_table[cols][:8]